RAG with webpage data extraction

In [17]:
# !pip install beautifulsoup4 lxml

In [18]:
import os
from pathlib import Path
from dotenv import load_dotenv

cwd = Path.cwd()
env_candidates = [
    cwd / ".env",
    cwd / "notebooks" / ".env",
    cwd.parent / "notebooks" / ".env",
]
env_file = next((p for p in env_candidates if p.exists()), None)
if env_file is None:
    raise FileNotFoundError("No .env found. Expected notebooks/.env with OPENAI_API_KEY.")
load_dotenv(env_file, override=True)

key = (os.getenv("OPENAI_API_KEY") or "").strip()
if not key or "paste_your_key" in key:
    raise ValueError(f"Set OPENAI_API_KEY in {env_file}.")

print(f"Loaded env from: {env_file}")
print("OPENAI_API_KEY configured: True")

Loaded env from: c:\Users\Girish Kulkarni\OneDrive\Documents\LLM_Testing\notebooks\.env
OPENAI_API_KEY configured: True


In [19]:
from langchain_openai import ChatOpenAI

MAX_OUTPUT_TOKENS = 10

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    max_tokens=MAX_OUTPUT_TOKENS,
)


In [20]:
# Load webpage(s). Put any URL you want to read here.
from langchain_community.document_loaders import WebBaseLoader

urls = [
    "https://www.descope.com/learn/post/mcp",
]

loader = WebBaseLoader(urls)
documents = loader.load()

print(f"Loaded {len(documents)} webpage(s)")
for doc in documents:
    print(doc.metadata.get("source"), "chars:", len(doc.page_content))

Loaded 1 webpage(s)
https://www.descope.com/learn/post/mcp chars: 27896


In [21]:
# Text Splitting

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, add_start_index=True)

all_split_docs = text_splitter.split_documents(documents)

len(all_split_docs)  # Total number of chunks after splitting the documents


38

In [22]:
# Embedding the chunks with OpenAI
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_one = embeddings.embed_query(all_split_docs[0].page_content)
vector_two = embeddings.embed_query(all_split_docs[1].page_content)

print(len(vector_one))
print(len(vector_two))

1536
1536


In [23]:
# Vector store (new folder: OpenAI embeddings are 1536-dim, old Ollama index was 768)
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=all_split_docs,
    embedding=embeddings,
    persist_directory="./chroma_langchain_db_openai",
    collection_name="webpage_rag_openai",
)

In [24]:
# Retrieve relevant chunks from the webpage(s)
from langchain_chroma import Chroma

vector_store = Chroma(
    persist_directory="./chroma_langchain_db_openai",
    embedding_function=embeddings,
    collection_name="webpage_rag_openai",
)

question = "What is retrieval-augmented generation?"
retrieved_docs = vector_store.similarity_search(question, k=3)

retrieved_docs

[Document(id='5eeb9ae5-c5a5-4229-ae78-b3b055e2c50b', metadata={'language': 'en', 'start_index': 4182, 'title': 'What Is the Model Context Protocol (MCP) and How It Works', 'source': 'https://www.descope.com/learn/post/mcp', 'description': 'Learn more about MCP, the open source protocol developed by Anthropic to provide LLMs and AI agents a standardized way to connect with external data sources and tools.'}, page_content='about recent data. This requires manually collecting information from various sources, feeding it into the LLM’s chat interface, and then extracting or applying the AI’s output elsewhere.\xa0While several models offer AI-powered web search, and Anthropic’s Claude 3.7 and 4 models boast a Computer Use feature, they still lack direct integration with knowledge stores and tools. Even as major platforms like OpenAI’s ChatGPT and Google’s Gemini add built-in app integrations, these remain platform-specific solutions rather than universal standards.For devs and enterprises, 

In [25]:
# Generate an answer from the retrieved context

context = "\n\n".join(doc.page_content for doc in retrieved_docs)

prompt = f"""Answer the question using only the context below.
If the answer is not present in the context, say: I don't know based on the webpage.

Context:
{context}

Question: {question}
Answer:"""

response = llm.invoke(prompt)
print(response.content)

I don't know based on the webpage.


In [26]:
# Retriever over the loaded webpage(s)
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

retriever.invoke("What is MCP explain me in 3 tokensa")

[Document(id='26d0aca8-c0ca-4183-96c7-1c004596fba0', metadata={'start_index': 1771, 'language': 'en', 'description': 'Learn more about MCP, the open source protocol developed by Anthropic to provide LLMs and AI agents a standardized way to connect with external data sources and tools.', 'source': 'https://www.descope.com/learn/post/mcp', 'title': 'What Is the Model Context Protocol (MCP) and How It Works'}, page_content="MCP standardization is foundational infrastructure for production AI applications. Understanding its architecture is essential for developers building connected AI systems.IdentipediaArrow LeftWhat Is the Model Context Protocol (MCP) and How It WorksJuly 28, 2026Copy linkShare on:Share on LinkedInShare on XShare on BluskyTable of ContentsLLM isolation & the NxM problemOpen table of contentsTable of ContentsLLM isolation & the NxM problemMCP architecture and core componentsHow MCP worksMCP client & server ecosystemSecurity considerations for MCP serversConclusionFAQs ab

In [27]:
# Full RetrievalQA implementation

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Full RetrievalQA implementation
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

question = "What is retrieval-augmented generation?"

prompt = ChatPromptTemplate.from_template("""Use only the context below to answer the question.
If the answer is not in the context, say: I don't know based on the documents.
Keep the answer concise and do not add unsupported details.

Context:
{context}

Question: {question}
Answer:""")


def format_documents(documents):
    return "\n\n".join(document.page_content for document in documents)


retrieval_qa_chain = (
    {
        "context": retriever | format_documents,
        "question": lambda value: value,
    }
    | prompt
    | llm
    | StrOutputParser()
)

answer = retrieval_qa_chain.invoke(question)
print(answer)

prompt = ChatPromptTemplate.from_template("""Use only the context below to answer the question.
If the answer is not in the context, say: I don't know based on the documents.
Keep the answer concise and do not add unsupported details.

Context:
{context}

Question: {question}
Answer:""")


def format_documents(documents):
    return "\n\n".join(document.page_content for document in documents)


retrieval_qa_chain = (
    {
        "context": retriever | format_documents,
        "question": lambda value: value,
    }
    | prompt
    | llm
    | StrOutputParser()
)

answer = retrieval_qa_chain.invoke(question)
print(answer)

I don't know based on the documents.
I don't know based on the documents.


In [28]:
# Geval


